# Math Prerequisites → Statistics | Chapter 1: Multivariate Gaussian Distribution

> **Why this chapter exists:** Probabilistic PCA (PPCA) and Robust PCA both rely on Gaussian assumptions. PPCA models data as drawn from a mixture of Gaussians perturbed by isotropic noise. Understanding the geometric shape of a multivariate Gaussian — its elliptical contours — is the same as understanding what the covariance matrix encodes, which is exactly what PCA diagonalizes.

---

## 1. Univariate Gaussian — Quick Recap

A random variable $X \sim \mathcal{N}(\mu, \sigma^2)$ has probability density:

$$p(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

- **$\mu$:** Mean — center of the distribution
- **$\sigma^2$:** Variance — width of the bell curve
- **68–95–99.7 rule:** 68% of data within $1\sigma$, 95% within $2\sigma$, 99.7% within $3\sigma$

---

## 2. The Multivariate Gaussian Distribution

A $d$-dimensional random vector $\vec{x} \in \mathbb{R}^d$ follows a **Multivariate Gaussian** $\vec{x} \sim \mathcal{N}(\vec{\mu}, \Sigma)$ with probability density:

$$p(\vec{x}) = \frac{1}{(2\pi)^{d/2} |\Sigma|^{1/2}} \exp\left(-\frac{1}{2}(\vec{x} - \vec{\mu})^T \Sigma^{-1} (\vec{x} - \vec{\mu})\right)$$

where:
- $\vec{\mu} \in \mathbb{R}^d$ — the **mean vector** (center of the distribution)
- $\Sigma \in \mathbb{R}^{d \times d}$ — the **covariance matrix** (shape and orientation)
- $|\Sigma|$ — determinant of $\Sigma$ (normalization constant)
- $\Sigma^{-1}$ — precision matrix (inverse covariance)

### **The Mahalanobis Distance**
The exponent contains the **Mahalanobis distance** squared:
$$\Delta^2 = (\vec{x} - \vec{\mu})^T \Sigma^{-1} (\vec{x} - \vec{\mu})$$

This is the Euclidean distance in the "standardized" space where the covariance is the identity. Points with the same Mahalanobis distance lie on an **ellipse** (in 2D) or **ellipsoid** (in $d$D).

The **shape of the ellipse** is entirely determined by $\Sigma$:
- **Axes of the ellipse** = eigenvectors of $\Sigma$ (= principal components!)
- **Lengths of the axes** = $\sqrt{\lambda_i}$ (standard deviations along each PC)

---

## 3. How $\Sigma$ Controls the Shape of the Gaussian

### **Case 1: Diagonal $\Sigma$ (Independent Features)**
$$\Sigma = \begin{bmatrix} \sigma_1^2 & 0 \\ 0 & \sigma_2^2 \end{bmatrix}$$
Ellipse aligned with the coordinate axes. Features are uncorrelated. PCA axes = original axes. No rotation needed.

### **Case 2: Isotropic $\Sigma = \sigma^2 I$ (Spherical)**
All variances equal, all covariances zero. The ellipse is a perfect circle. Every direction has the same variance. PCA makes no useful distinction between directions.

### **Case 3: General $\Sigma$ (Correlated Features)**
Non-zero off-diagonal entries → ellipse tilted relative to the coordinate axes. The **eigenvectors of $\Sigma$** are the axes of the tilted ellipse. **This is exactly what PCA finds** — it rotates to align the coordinate axes with the ellipse axes.

### **A Key Visualization Insight**
```
Original space:         After PCA rotation (PC space):

    y                       PC2
    |  ╱‾‾╲                  |  ║‾‾║
    | ╱ 🔴 ╲                 |  ║ 🔴║
    |╱      ╲                |  ║   ║
    └──────── x              └──╨───╨── PC1
    Tilted ellipse           Axis-aligned ellipse
    Σ has off-diagonal       Λ = diag(λ₁, λ₂)
```

PCA finds the rotation $W$ such that $Z = XW$ has a diagonal covariance matrix $\Lambda$. In PC space, features are uncorrelated.

---

## 4. Why Gaussian Matters for PPCA

Probabilistic PCA models the data generatively:
$$\vec{z} \sim \mathcal{N}(\vec{0}, I_k) \quad \text{(latent / low-dim)}$$
$$\vec{x} | \vec{z} \sim \mathcal{N}(W\vec{z} + \vec{\mu},\ \sigma^2 I_d) \quad \text{(observed / high-dim)}$$

- $\vec{z} \in \mathbb{R}^k$: the latent (low-dimensional) code — what we want to find.
- $W \in \mathbb{R}^{d \times k}$: the loading matrix — maps latent to observed.
- $\sigma^2$: isotropic Gaussian noise in every observed dimension.

**Marginalizing over $\vec{z}$** gives the observed distribution:
$$\vec{x} \sim \mathcal{N}(\vec{\mu},\ WW^T + \sigma^2 I_d)$$

The covariance of observed data is $C = WW^T + \sigma^2 I_d$:
- $WW^T$ = low-rank structure (the signal)
- $\sigma^2 I$ = isotropic noise

This is the PPCA generative model in Chapter 3 of the PCA series. As $\sigma^2 \to 0$: PPCA → standard PCA.

---

## 5. PCA Decorrelates — And for Gaussians, That Means Independence

For **any** distribution: uncorrelated $\neq$ independent. Two variables can have zero covariance but still be dependent.

**Exception: For multivariate Gaussians, uncorrelated = independent.**

Proof sketch: If $\vec{x} \sim \mathcal{N}(\vec{\mu}, \Sigma)$ and $\Sigma$ is diagonal, then the joint density factors:
$$p(\vec{x}) = \prod_{i=1}^d p_i(x_i) = \prod_{i=1}^d \mathcal{N}(\mu_i, \sigma_i^2)$$

**Consequence for PCA on Gaussian data:**
After PCA rotation, $Z = XW$:
- Covariance of $Z$ = $\Lambda$ (diagonal) → PC scores are **uncorrelated**
- If original data was Gaussian → PC scores are **statistically independent**
- Each PC can be modeled and interpreted independently

This is why PPCA and Factor Analysis assume Gaussian noise — it gives the strongest statistical separation between components.

> ⚠️ For **non-Gaussian** data, PCA still decorrelates (diagonal $\Lambda$) but does **not** give statistical independence. Independent Component Analysis (ICA) is needed for true independence in non-Gaussian cases.

---

## 6. Implementation Playground

In [ ]:
# ─── CELL 1: Visualize Multivariate Gaussians — Shape of Σ ────────────────────
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

def cov_to_ellipse(ax, cov, mu, n_std=2.0, **kwargs):
    """Draw an ellipse representing the covariance matrix."""
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    # Sort descending
    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]
    angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(eigenvalues)
    e = Ellipse(xy=mu, width=width, height=height, angle=angle, **kwargs)
    ax.add_patch(e)
    # Draw eigenvector arrows
    colors = ['red', 'blue']
    for i, (ev, lam) in enumerate(zip(eigenvectors.T, eigenvalues)):
        ax.quiver(*mu, ev[0]*np.sqrt(lam)*n_std, ev[1]*np.sqrt(lam)*n_std,
                  color=colors[i], scale_units='xy', scale=1, width=0.025, zorder=5)

np.random.seed(42)
N = 400
mu = np.array([0., 0.])

configs = [
    (np.array([[1, 0], [0, 1]]),      'Isotropic (σ²I)\nSphere: equal variance all directions'),
    (np.array([[3, 0], [0, 1]]),      'Diagonal: different variances\nEllipse aligned with axes'),
    (np.array([[3, 1.5], [1.5, 2]]), 'Correlated features\nTilted ellipse: PCA rotates this'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (cov, title) in zip(axes, configs):
    X = np.random.multivariate_normal(mu, cov, N)
    ax.scatter(X[:,0], X[:,1], alpha=0.25, s=6, c='steelblue')
    cov_to_ellipse(ax, cov, mu, n_std=2, fill=False, edgecolor='orange', lw=2.5, label='2σ ellipse')
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
    ax.set_aspect('equal'); ax.grid(alpha=0.3)
    eigenvalues, _ = np.linalg.eigh(cov)
    ax.set_title(f'{title}\nλ₁={eigenvalues[-1]:.1f}, λ₂={eigenvalues[0]:.1f}', fontsize=10)
    ax.axhline(0, c='k', lw=0.5); ax.axvline(0, c='k', lw=0.5)
    ax.text(-4.5, 4, 'Red=PC1, Blue=PC2', fontsize=8, color='gray')

plt.suptitle('Multivariate Gaussian: Shape Controlled by Σ\nEigenvectors = Ellipse Axes = Principal Components', 
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ─── CELL 2: PPCA Generative Model — Sample from It & Verify ─────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

np.random.seed(0)
N = 500     # number of samples
k = 2       # latent dimensions
d = 10      # observed dimensions
sigma2 = 0.5  # noise variance

# Generate PPCA model
W = np.random.randn(d, k)  # loading matrix (d x k)
mu = np.zeros(d)           # zero mean for simplicity

# Sample from PPCA generative model:
z = np.random.randn(N, k)          # latent codes ~ N(0, I_k)
eps = np.sqrt(sigma2) * np.random.randn(N, d)  # noise ~ N(0, σ²I)
X = z @ W.T + mu + eps             # observed data ~ N(μ, WW^T + σ²I)

# Verify: covariance of X should be ~ WW^T + σ²I
C_theoretical = W @ W.T + sigma2 * np.eye(d)
C_empirical = np.cov(X.T)

print(f"Theoretical covariance eigenvalues (first 5): {np.sort(np.linalg.eigvalsh(C_theoretical))[::-1][:5].round(3)}")
print(f"Empirical covariance eigenvalues (first 5):   {np.sort(np.linalg.eigvalsh(C_empirical))[::-1][:5].round(3)}")

# PCA should recover first 2 components (the latent dims)
pca = PCA(n_components=d)
pca.fit(X)

print(f"\nExplained variance ratio by component: {pca.explained_variance_ratio_.round(3)}")
print(f"First {k} PCs capture: {pca.explained_variance_ratio_[:k].sum()*100:.1f}% (signal)")
print(f"Remaining noise variance ≈ σ² from dims {k+1}+: {pca.explained_variance_[k:].mean():.3f} (≈ {sigma2})")

# Scree plot: elbow at k=2 (the latent dimension)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(range(1, d+1), pca.explained_variance_, color='steelblue', alpha=0.7)
ax1.axvline(k + 0.5, color='red', ls='--', lw=2, label=f'True latent dim k={k}')
ax1.set_xlabel('Component'); ax1.set_ylabel('Explained Variance (eigenvalue)')
ax1.set_title('Scree Plot: Elbow at k=2'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.scatter(X[:,0], X[:,1], alpha=0.3, s=8, c='gray', label='Observed data (dim 1 vs 2)')
Z = pca.transform(X)
ax2.scatter(Z[:,0], Z[:,1], alpha=0.3, s=8, c='steelblue', label='PC1 vs PC2 scores')
ax2.set_xlabel('Feature 1 / PC1'); ax2.set_ylabel('Feature 2 / PC2')
ax2.set_title('PPCA: Original vs PC Space'); ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()